# Retrieval

In [1]:
import numpy as np
import pandas as pd

from rlt.data import load, product_text
from rlt.metrics import gains, ndcg_at, rcap_at
from rlt.retrieve import bm25_search

products, queries, labels = load()
query_ids = queries.index.to_numpy()
fields = {
    "title": ["product_name"],
    "full": ["product_name", "product_class", "category hierarchy", "product_description", "product_features"],
}

## BM25

In [2]:
runs = {
    f"bm25 {name}": bm25_search(product_text(products, columns), queries["query"])
    for name, columns in fields.items()
}

## Results

Mean over queries. Rcap@k = relevant products in the top k / min(relevant products, k). Queries with no relevant product are left out (101 for Exact, 1 for Exact + Partial).

In [3]:
def evaluate(ranked_ids):
    ranked_gains = gains(ranked_ids, query_ids, labels)
    return {
        "nDCG@10": np.nanmean(ndcg_at(ranked_gains, query_ids, labels, 10)),
        "Rcap@100 Exact": np.nanmean(rcap_at(ranked_gains, query_ids, labels, 100, 2)),
        "Rcap@100 Exact + Partial": np.nanmean(rcap_at(ranked_gains, query_ids, labels, 100, 1)),
        "Rcap@1000 Exact + Partial": np.nanmean(rcap_at(ranked_gains, query_ids, labels, 1000, 1)),
    }


pd.DataFrame({name: evaluate(ids) for name, (ids, scores) in runs.items()}).T.round(3)

,nDCG@10,Rcap@100 Exact,Rcap@100 Exact + Partial,Rcap@1000 Exact + Partial
bm25 title,0.706,0.645,0.716,0.851
bm25 full,0.695,0.719,0.668,0.849
